# adminlineage Gemini 3.1 Flash-Lite notebook

This notebook is the main Gemini 3.1 Flash-Lite workflow for running `adminlineage` on your own files.

You only need to edit the input settings cell near the top, then run the notebook from top to bottom.

## Before you start

1. Install the package in this repo:

```bash
python -m pip install -e .[dev,io]
```

2. Put your Gemini key in `.env` if you want to run the LLM step:

```text
GEMINI_API_KEY=your_api_key_here
```

The package will search for `.env` from the configured repo root below, so the run does not depend on the kernel's working directory.

3. Replace the placeholder paths and field names in the next code cell with your real dataset details.

In [1]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd

import adminlineage
from adminlineage.utils import sanitize_name


## Load the datasets

In [3]:
# Run this after you update the file paths above.
df_from = pd.read_stata("C:\\Users\\Taha rice\\Downloads\\Shrug state and district.dta")
df_to = pd.read_csv("C:\\Users\\Taha rice\\Downloads\\district_25.csv")

print(f"Loaded df_from with shape: {df_from.shape}")
print(f"Loaded df_to with shape: {df_to.shape}")


Loaded df_from with shape: (631, 2)
Loaded df_to with shape: (3361, 1)


In [4]:
print("df_from columns:")
print(df_from.columns.tolist())

print("\ndf_to columns:")
print(df_to.columns.tolist())


df_from columns:
['state_name', 'district_name']

df_to columns:
['District']


## Edit these values

Update only this cell first. After that, you can run the rest of the notebook in order.

In [ ]:
# Country and period labels.
country = "India"
year_from = 2011
year_to = 2025

# Name columns to match.
map_col_from = "district_name"
map_col_to = "District"

# Exact-match columns are optional.
# Leave this as [] if you want global matching.
#exact_match = ["state", "district"]

# ID columns are optional. Use None if you do not have stable IDs.
#id_col_from = "unit_id"
#id_col_to = "unit_id"

# Extra context columns are optional.
extra_context_cols = [
    (
        "there are districts that were formed, split, or renamed between this period. "
        "2025 is just a placeholder for the most recent year for this dataset, "
        "and 2011 is just a placeholder for the most recent census year."
    )
]

# Start with relationship='auto' unless you want to force one mode.
# Allowed values: 'auto', 'father_to_father', 'father_to_child', 'child_to_father', 'child_to_child'
relationship = "auto"

# Keep evidence=False and reason=False at first to save tokens.
evidence = False
reason = False

# LLM settings.
# string_exact_match_prune options: "none", "from", "to"
# When this is "from" or "to", adminlineage also runs one bounded second-stage
# rescue pass over merge==only_in_from or merge==only_in_to rows after first-pass output.
# Gemini calls honor batch_size.
# If a multi-row request fails, adminlineage retries in smaller batches.
model = "gemini-3.1-flash-lite-preview"
string_exact_match_prune = "to"
gemini_api_key_env = "GEMINI_API_KEY"
batch_size = 5
max_candidates = 6
seed = 42
temperature = 0.75
enable_google_search = True
request_timeout_seconds = 90
max_first_batch_minutes = 5
max_batch_stall_minutes = 20

# adminlineage notebook runs are usually editable installs, so this points back to the repo.
package_root = Path(adminlineage.__file__).resolve().parents[2]
env_search_dir = package_root if (package_root / ".env").exists() else Path.cwd()
run_label = "adminlineage_notebook_flash_lite_grounded"
output_dir = package_root / "outputs" / run_label
replay_enabled = True
replay_store_dir = package_root / ".adminlineage_replay"
force_live_cleanup = False
run_name = f"{sanitize_name(country)}_{year_from}_{year_to}_{sanitize_name(map_col_from)}"
run_dir = output_dir / run_name

if force_live_cleanup:
    previous_metadata_path = run_dir / "run_metadata.json"
    previous_replay_key = None
    if previous_metadata_path.exists():
        previous_metadata = json.loads(previous_metadata_path.read_text(encoding="utf-8"))
        previous_replay_key = previous_metadata.get("replay_key")
    shutil.rmtree(run_dir, ignore_errors=True)
    if previous_replay_key:
        shutil.rmtree(replay_store_dir / previous_replay_key, ignore_errors=True)
    print(f"Removed prior run directory: {run_dir}")
    if previous_replay_key:
        print(f"Removed replay bundle: {replay_store_dir / previous_replay_key}")

string_exact_match_prune = os.getenv(
    "ADMINLINEAGE_NOTEBOOK_STRING_EXACT_MATCH_PRUNE",
    string_exact_match_prune,
)
batch_size = int(os.getenv("ADMINLINEAGE_NOTEBOOK_BATCH_SIZE", str(batch_size)))

print(f"Using .env search root: {env_search_dir}")
print(f"Notebook output base dir: {output_dir}")
print(f"Resolved run directory: {run_dir}")
print(f"Replay store: {replay_store_dir}")


Using .env search root: C:\Users\Taha rice\Desktop\projects\AdminLineageAI
Notebook output base dir: C:\Users\Taha rice\Desktop\projects\AdminLineageAI\outputs\adminlineage_notebook_flash_lite_grounded
Resolved run directory: C:\Users\Taha rice\Desktop\projects\AdminLineageAI\outputs\adminlineage_notebook_flash_lite_grounded\india_2011_2025_district-name
Replay store: C:\Users\Taha rice\Desktop\projects\AdminLineageAI\.adminlineage_replay


## Validate the inputs

This checks whether the key columns you configured are present and whether the exact-match setup looks usable.

In [6]:
validation = adminlineage.validate_inputs(
    df_from,
    df_to,
    country=country,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
)

validation


{'valid': True,
 'errors': [],
 'warnings': ['No exact_match columns supplied; candidate generation will run globally and may increase false matches.',
  "Collapsed 6 duplicate df_from rows using match key [district_name]; 625 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (district_name='hamirpur'); (district_name='bilaspur'); (district_name='pratapgarh').",
  "Collapsed 2750 duplicate df_to rows using match key [District]; 611 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (District='north and middle andaman'); (District='south andamans'); (District='visakhapatanam')."],
 'map_col_to': 'District',
 'from_rows_input': 631,
 'from_rows_effective': 625,
 'to_rows_input': 3361,
 'to_rows_effective': 611,
 'collapse_repor

## Preview the candidate plan

This does not call the LLM. It gives you a quick look at grouping and candidate counts before you run the full job.

In [7]:
preview = adminlineage.preview_plan(
    df_from,
    df_to,
    country=country,
    year_from=year_from,
    year_to=year_to,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
    extra_context_cols=extra_context_cols,
    string_exact_match_prune=string_exact_match_prune,
    max_candidates=max_candidates,
)

preview


{'valid': True,
 'country': 'India',
 'year_from': 2011,
 'year_to': 2025,
 'from_rows_input': 631,
 'from_rows_effective': 625,
 'to_rows_input': 3361,
 'to_rows_effective': 611,
 'from_rows': 139,
 'to_rows': 125,
 'exact_match': [],
 'string_exact_match_prune': 'to',
 'exact_string_matches': 486,
 'groups': {'__all__': 139},
 'max_candidates': 6,
 'avg_candidates': 6.0,
 'warnings': ['No exact_match columns supplied; candidate generation will run globally and may increase false matches.',
  "Collapsed 6 duplicate df_from rows using match key [district_name]; 625 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (district_name='hamirpur'); (district_name='bilaspur'); (district_name='pratapgarh').",
  "Collapsed 2750 duplicate df_to rows using match key [District]; 611 unique rows remain. Duplicate values were ignored and only the first row per key was k

## Run the package

Run this cell only after the validation and preview look reasonable. This is the step that makes the LLM call and writes output files.

In [8]:
crosswalk_df, metadata = adminlineage.build_evolution_key(
    df_from,
    df_to,
    country=country,
    year_from=year_from,
    year_to=year_to,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
    extra_context_cols=extra_context_cols,
    relationship=relationship,
    string_exact_match_prune=string_exact_match_prune,
    evidence=evidence,
    reason=reason,
    model=model,
    gemini_api_key_env=gemini_api_key_env,
    batch_size=batch_size,
    max_candidates=max_candidates,
    output_dir=output_dir,
    seed=seed,
    temperature=temperature,
    enable_google_search=enable_google_search,
    request_timeout_seconds=request_timeout_seconds,
    env_search_dir=env_search_dir,
    replay_enabled=replay_enabled,
    replay_store_dir=replay_store_dir,
)


2026-04-09T20:52:50 | INFO | run_id=185a546316abef97 stage=start country=India years=2011->2025
2026-04-09T20:52:50 | INFO | run_id=185a546316abef97 stage=replay miss replay_key=c07151e09bb4c4579aae5bb4
2026-04-09T20:52:50 | INFO | run_id=185a546316abef97 stage=grounding mode=single_pass_structured_json enabled=true
2026-04-09T20:52:51 | INFO | run_id=185a546316abef97 stage=resume exact_mode=to exact_matched=486 completed=486 pending=139
2026-04-09T20:52:51 | INFO | run_id=185a546316abef97 stage=adjudication match_stage=ai batch=1 size=5
2026-04-09T20:53:00 | INFO | run_id=185a546316abef97 stage=adjudication match_stage=ai batch=2 size=5
2026-04-09T20:54:41 | INFO | run_id=185a546316abef97 stage=adjudication match_stage=ai batch=3 size=5
2026-04-09T20:57:44 | INFO | run_id=185a546316abef97 stage=adjudication match_stage=ai batch=4 size=5
2026-04-09T20:58:01 | INFO | run_id=185a546316abef97 stage=adjudication match_stage=ai batch=5 size=5
2026-04-09T20:58:05 | INFO | run_id=185a546316ab

## Inspect the output

In [ ]:
crosswalk_df.head()


In [ ]:
print(crosswalk_df.columns.tolist())


In [ ]:
replay_summary = {
    "execution_mode": metadata.get("execution_mode"),
    "replay_hit": metadata.get("replay_hit"),
    "replay_key": metadata.get("replay_key"),
    "replayed_from_run_id": metadata.get("replayed_from_run_id"),
}
second_stage_summary = {
    "second_stage_attempted_rows": metadata.get("counts", {}).get("second_stage_attempted_rows"),
    "second_stage_rewritten_rows": metadata.get("counts", {}).get("second_stage_rewritten_rows"),
    "second_stage_failed_rows": metadata.get("counts", {}).get("second_stage_failed_rows"),
}
print(replay_summary)
print(second_stage_summary)
metadata


In [ ]:
artifacts = metadata.get("artifacts", {})
artifacts


## Optional: read back the saved crosswalk file

This is useful if you want to confirm what was written to disk.

In [ ]:
csv_path = artifacts.get("evolution_key_csv")
if csv_path:
    saved_crosswalk = pd.read_csv(csv_path)
    saved_crosswalk.head()
else:
    print("No CSV artifact was recorded for this run.")
